# 04e — Text vs Crisis Head-to-Head

**Input:** `../data/processed/pm_day_features.csv`, `../data/processed/embeddings.npy`
**Output:** `../outputs/tables/headtohead_summary.csv`

**Description:**
- Direct comparison: text-only vs crisis-total-only vs combined
- Answers: "Is text better than just asking about crisis?"
- Also compares minimal baseline (crisis_PM only) vs full baseline (crisis + other PM ratings)
- Binary outcomes with Logistic Regression, GroupKFold CV
- Pearson/Spearman correlations between OOF probabilities and observed DSI
- Matches original cells 24, 32

In [2]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score
from scipy.stats import pearsonr, spearmanr

# =========================
# CONFIG
# =========================
DATA_PATH = os.path.join("..", "data", "processed", "pm_day_features.csv")
EMBED_PATH = os.path.join("..", "data", "processed", "embeddings.npy")
OUT_DIR = os.path.join("..", "outputs", "tables")
os.makedirs(OUT_DIR, exist_ok=True)

PID_COL = "expiwell_id_clean"
CRISIS_COL = "crisis_PM_from_full"

N_SPLITS = 5
N_PCS = 20
C_LOGIT = 1.0
RANDOM_SEED = 7
np.random.seed(RANDOM_SEED)

OUTCOMES = ["any_risk_total_gt0", "moderate_total_ge2", "high_any_item_eq3"]

In [3]:
# =========================
# LOAD
# =========================
pm_day = pd.read_csv(DATA_PATH)
X_text = np.load(EMBED_PATH)
assert X_text.shape[0] == len(pm_day)

groups = pm_day[PID_COL].astype(str).values

# Build numeric baselines
# Minimal: crisis_PM only
X_crisis_only = pm_day[[CRISIS_COL]].astype(float).fillna(0).values

# Full: crisis + other PM ratings
num_candidates = [c for c in pm_day.columns if c.startswith("dailyPM__") and pd.api.types.is_numeric_dtype(pm_day[c])]
X_num_df = pm_day[[CRISIS_COL] + num_candidates].copy().apply(pd.to_numeric, errors="coerce")
X_num_df = X_num_df.dropna(axis=1, how="all")
X_full_num = X_num_df.fillna(X_num_df.mean()).values.astype(float)
nan_cols = np.any(np.isnan(X_full_num), axis=0)
if nan_cols.any():
    X_full_num = X_full_num[:, ~nan_cols]

print("Loaded data:", pm_day.shape)
print("Crisis-only features:", X_crisis_only.shape[1])
print("Full numeric features:", X_full_num.shape[1])

Loaded data: (2511, 73)
Crisis-only features: 1
Full numeric features: 15


In [6]:
# =========================
# HEAD-TO-HEAD OOF FUNCTION
# =========================
def oof_logit(X, y, groups, label=""):
    g = np.asarray(groups).astype(str)
    gkf = GroupKFold(n_splits=min(N_SPLITS, len(np.unique(g))))
    p = np.zeros(len(y), dtype=float)
    for tr, te in gkf.split(X, y, groups=g):
        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("logit", LogisticRegression(C=C_LOGIT, max_iter=8000, solver="lbfgs"))
        ])
        pipe.fit(X[tr], y[tr])
        p[te] = pipe.predict_proba(X[te])[:, 1]
    auroc = roc_auc_score(y, p)
    auprc = average_precision_score(y, p)
    return p, auroc, auprc


def oof_logit_with_pca(X_text, X_num, y, groups, n_pcs=20, label=""):
    g = np.asarray(groups).astype(str)
    gkf = GroupKFold(n_splits=min(N_SPLITS, len(np.unique(g))))
    p = np.zeros(len(y), dtype=float)
    for tr, te in gkf.split(X_text, y, groups=g):
        pca = PCA(n_components=min(n_pcs, X_text.shape[1]), random_state=RANDOM_SEED)
        Ttr = pca.fit_transform(X_text[tr])
        Tte = pca.transform(X_text[te])

        if X_num is not None:
            Xtr = np.hstack([X_num[tr], Ttr])
            Xte = np.hstack([X_num[te], Tte])
        else:
            Xtr, Xte = Ttr, Tte

        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("logit", LogisticRegression(C=C_LOGIT, max_iter=8000, solver="lbfgs"))
        ])
        pipe.fit(Xtr, y[tr])
        p[te] = pipe.predict_proba(Xte)[:, 1]
    auroc = roc_auc_score(y, p)
    auprc = average_precision_score(y, p)
    return p, auroc, auprc

In [8]:
# =========================
# RUN HEAD-TO-HEAD
# =========================
all_rows = []

for outcome in OUTCOMES:
    y = pm_day[outcome].astype(int).values
    pos = int(y.sum())

    if pos < 20:
        print(f"Skipping {outcome}: too few positives ({pos})")
        continue

    print(f"\n{'='*60}")
    print(f"Outcome: {outcome} | N={len(y)} | pos={pos} ({y.mean():.3f})")
    print(f"{'='*60}")

    # 1) Crisis-only (minimal baseline)
    p_crisis, auroc_crisis, auprc_crisis = oof_logit(X_crisis_only, y, groups)
    print(f"  Crisis-only:     AUROC={auroc_crisis:.3f} AUPRC={auprc_crisis:.3f}")

    # 2) Full numeric baseline
    p_fullnum, auroc_fullnum, auprc_fullnum = oof_logit(X_full_num, y, groups)
    print(f"  Full numeric:    AUROC={auroc_fullnum:.3f} AUPRC={auprc_fullnum:.3f}")

    # 3) Text-only (no numeric)
    p_textonly, auroc_textonly, auprc_textonly = oof_logit_with_pca(X_text, None, y, groups, N_PCS)
    print(f"  Text-only:       AUROC={auroc_textonly:.3f} AUPRC={auprc_textonly:.3f}")

    # 4) Crisis + text
    p_crisis_text, auroc_ct, auprc_ct = oof_logit_with_pca(X_text, X_crisis_only, y, groups, N_PCS)
    print(f"  Crisis + text:   AUROC={auroc_ct:.3f} AUPRC={auprc_ct:.3f}")

    # 5) Full numeric + text
    p_full_text, auroc_ft, auprc_ft = oof_logit_with_pca(X_text, X_full_num, y, groups, N_PCS)
    print(f"  Full + text:     AUROC={auroc_ft:.3f} AUPRC={auprc_ft:.3f}")

    # Pearson/Spearman correlations with continuous DSI
    if "dsi_PM_total" in pm_day.columns:
        dsi = pm_day["dsi_PM_total"].values
        mask_corr = np.isfinite(dsi)
        if mask_corr.sum() > 50:
            r_text, _ = pearsonr(dsi[mask_corr], p_textonly[mask_corr])
            rho_text, _ = spearmanr(dsi[mask_corr], p_textonly[mask_corr])
            r_full, _ = pearsonr(dsi[mask_corr], p_full_text[mask_corr])
            print(f"  Correlation with DSI: text-only r={r_text:.3f} rho={rho_text:.3f} | full r={r_full:.3f}")

    for tag, auroc, auprc in [
        ("crisis_only", auroc_crisis, auprc_crisis),
        ("full_numeric", auroc_fullnum, auprc_fullnum),
        ("text_only", auroc_textonly, auprc_textonly),
        ("crisis_plus_text", auroc_ct, auprc_ct),
        ("full_plus_text", auroc_ft, auprc_ft),
    ]:
        all_rows.append({"outcome": outcome, "model": tag, "n": len(y), "pos": pos,
                         "AUROC": auroc, "AUPRC": auprc})


Outcome: any_risk_total_gt0 | N=2511 | pos=940 (0.374)
  Crisis-only:     AUROC=0.798 AUPRC=0.697
  Full numeric:    AUROC=0.845 AUPRC=0.766
  Text-only:       AUROC=0.700 AUPRC=0.554
  Crisis + text:   AUROC=0.810 AUPRC=0.709
  Full + text:     AUROC=0.846 AUPRC=0.753
  Correlation with DSI: text-only r=0.354 rho=0.361 | full r=0.609

Outcome: moderate_total_ge2 | N=2511 | pos=877 (0.349)
  Crisis-only:     AUROC=0.807 AUPRC=0.691
  Full numeric:    AUROC=0.855 AUPRC=0.758
  Text-only:       AUROC=0.703 AUPRC=0.536
  Crisis + text:   AUROC=0.815 AUPRC=0.696
  Full + text:     AUROC=0.852 AUPRC=0.737
  Correlation with DSI: text-only r=0.356 rho=0.360 | full r=0.611

Outcome: high_any_item_eq3 | N=2511 | pos=99 (0.039)
  Crisis-only:     AUROC=0.628 AUPRC=0.064
  Full numeric:    AUROC=0.685 AUPRC=0.065
  Text-only:       AUROC=0.750 AUPRC=0.193
  Crisis + text:   AUROC=0.732 AUPRC=0.236
  Full + text:     AUROC=0.756 AUPRC=0.167
  Correlation with DSI: text-only r=0.166 rho=0.175 | f

In [10]:
# =========================
# SAVE
# =========================
summary = pd.DataFrame(all_rows)
summary_path = os.path.join(OUT_DIR, "headtohead_summary.csv")
summary.to_csv(summary_path, index=False)

print("\n" + "=" * 60)
print("HEAD-TO-HEAD SUMMARY")
print("=" * 60)
print(summary.to_string(index=False))
print("\nSaved:", summary_path)


HEAD-TO-HEAD SUMMARY
           outcome            model    n  pos    AUROC    AUPRC
any_risk_total_gt0      crisis_only 2511  940 0.797748 0.697425
any_risk_total_gt0     full_numeric 2511  940 0.845295 0.766268
any_risk_total_gt0        text_only 2511  940 0.700117 0.553600
any_risk_total_gt0 crisis_plus_text 2511  940 0.809554 0.708866
any_risk_total_gt0   full_plus_text 2511  940 0.846284 0.752772
moderate_total_ge2      crisis_only 2511  877 0.807404 0.691219
moderate_total_ge2     full_numeric 2511  877 0.854619 0.757892
moderate_total_ge2        text_only 2511  877 0.702963 0.535567
moderate_total_ge2 crisis_plus_text 2511  877 0.815396 0.696154
moderate_total_ge2   full_plus_text 2511  877 0.852262 0.737135
 high_any_item_eq3      crisis_only 2511   99 0.627714 0.063683
 high_any_item_eq3     full_numeric 2511   99 0.685432 0.064900
 high_any_item_eq3        text_only 2511   99 0.749711 0.193187
 high_any_item_eq3 crisis_plus_text 2511   99 0.731739 0.236199
 high_any_item_eq3